# Stepwise, all-subsets and composite-index regression

This notebook extends the regression of notebook 7. It answers three questions:

1. **Hierarchical (stepwise) regression:** how much does each group of characteristics add to the explained variation (ΔR²), and how do the coefficients of the other variables change when a new group is added? A coefficient that shrinks or changes sign when another variable enters shows that the two variables are interdependent.
2. **All-subsets regression:** which combination of variables explains the most variation, and how much is gained by adding more variables?
3. **Composite indices:** strongly correlated pairs (|r| ≥ 0.75 in notebook 6) are combined into one index instead of dropping one of them.

The variables are entered in an order chosen in advance on theoretical grounds, not selected automatically on p-values; automatic selection on p-values tends to find chance results.

**Input:** `data/processed/all_variables_per_municipality.csv` (notebook 6).

**Outputs:** `data/processed/stepwise_results.csv`, `data/processed/all_subsets_results.csv`, `data/processed/composite_model_results.csv`, `figures/figure_stepwise_coefficients.png`, `figures/figure_all_subsets.png`

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

PROCESSED = "../data/processed/"
FIGURES = "../figures/"

## 1. Load the data

All models in this notebook are estimated on the **same** municipalities (complete cases), so that R² values can be compared between models. All variables are standardised (z-scores), as in notebook 7, so coefficients (β) are directly comparable.

In [ ]:
df = pd.read_csv(PROCESSED + "all_variables_per_municipality.csv")
df["log_population_density"] = np.log(df["bevolkingsdichtheid"])

TARGET = "laadpunten_per_1000_huishoudens"
VARIABLES = {
    "med_gestandaardiseerd_inkomen_keur": "Median income",
    "hoogopgeleid_pct": "Highly educated (%)",
    "ouderen_65plus_pct": "Residents 65+ (%)",
    "linkse_stemmen_pct": "Left-wing vote share (%)",
    "log_population_density": "Population density (log)",
    "meergezinswoning_pct": "Apartments (%)",
}

data = df[["gemeente", TARGET] + list(VARIABLES)].dropna()
z = data[[TARGET] + list(VARIABLES)]
z = (z - z.mean()) / z.std()

print("Municipalities used:", len(data), "of", len(df))
print("Left out (missing income):", sorted(set(df["gemeente"]) - set(data["gemeente"])))

In [ ]:
def fit(predictors):
    """OLS on z-scores. Robust (HC3) standard errors for the coefficients;
    the ordinary fit is kept for the F-test on ΔR²."""
    X = sm.add_constant(z[list(predictors)])
    ols = sm.OLS(z[TARGET], X).fit()
    robust = ols.get_robustcov_results(cov_type="HC3")
    params = pd.Series(robust.params, index=X.columns)
    pvalues = pd.Series(robust.pvalues, index=X.columns)
    return ols, params, pvalues


def max_vif(predictors):
    if len(predictors) < 2:
        return 1.0
    X = sm.add_constant(z[list(predictors)]).values
    return max(variance_inflation_factor(X, i) for i in range(1, X.shape[1]))


def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

## 2. Hierarchical regression

The groups of characteristics are added one at a time, in the order of the research questions:

| Step | Added | Group (sub-question) |
|---|---|---|
| 1 | Median income | Socioeconomic (SQ2) |
| 2 | Highly educated | Socioeconomic (SQ2) |
| 3 | Residents 65+ | Demographic (SQ3) |
| 4 | Left-wing vote share | Political (SQ4) |
| 5 | Population density (log) | Need (SQ5) |
| 6 | Apartments | Need (SQ5) |

For each step we report R², adjusted R², the increase in R² (ΔR²) with an F-test (is the increase significant?), and the β of every variable in the model.

In [ ]:
ORDER = ["med_gestandaardiseerd_inkomen_keur", "hoogopgeleid_pct", "ouderen_65plus_pct",
         "linkse_stemmen_pct", "log_population_density", "meergezinswoning_pct"]


def hierarchical(order):
    rows, previous = [], None
    for k in range(1, len(order) + 1):
        ols, params, pvalues = fit(order[:k])
        row = {"Step": k, "Added": VARIABLES[order[k - 1]],
               "R²": ols.rsquared, "Adj. R²": ols.rsquared_adj,
               "ΔR²": ols.rsquared - (previous.rsquared if previous else 0),
               "p (ΔR²)": ols.compare_f_test(previous)[1] if previous else ols.f_pvalue,
               "Max VIF": max_vif(order[:k])}
        for var in order[:k]:
            row[VARIABLES[var]] = f"{params[var]:+.2f}{stars(pvalues[var])}"
        rows.append(row)
        previous = ols
    return pd.DataFrame(rows).set_index("Step")


stepwise = hierarchical(ORDER)
stepwise.to_csv(PROCESSED + "stepwise_results.csv")
stepwise.style.format({"R²": "{:.3f}", "Adj. R²": "{:.3f}", "ΔR²": "{:+.3f}",
                       "p (ΔR²)": "{:.3g}", "Max VIF": "{:.2f}"}, na_rep="")

*β = standardised coefficient, robust (HC3) significance: \* p < 0.05, \*\* p < 0.01, \*\*\* p < 0.001. An empty cell means the variable is not yet in the model.*

The figure below shows how the β of each variable changes as the other variables are added. A line that moves towards zero means that part of the earlier effect actually ran through a variable that was added later.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for k, var in enumerate(ORDER):
    betas = []
    for step in range(1, len(ORDER) + 1):
        if step <= k:
            betas.append(np.nan)
        else:
            betas.append(fit(ORDER[:step])[1][var])
    ax.plot(range(1, len(ORDER) + 1), betas, marker="o", label=VARIABLES[var])

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(1, len(ORDER) + 1))
ax.set_xticklabels([f"{i}\n+ {VARIABLES[v].split(' (')[0]}" for i, v in enumerate(ORDER, 1)], fontsize=8)
ax.set_ylabel("Standardised coefficient β")
ax.set_title(f"Coefficients per step of the hierarchical regression (N = {len(data)})")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(FIGURES + "figure_stepwise_coefficients.png", dpi=200, bbox_inches="tight")
plt.show()

### 2b. Order check: need first

ΔR² depends on the order in which variables are added. To see how much the socioeconomic characteristics add **on top of** need, the order is reversed: need first, then the other groups.

In [ ]:
ORDER_NEED_FIRST = ["log_population_density", "meergezinswoning_pct", "ouderen_65plus_pct",
                    "linkse_stemmen_pct", "hoogopgeleid_pct", "med_gestandaardiseerd_inkomen_keur"]
stepwise_need_first = hierarchical(ORDER_NEED_FIRST)
stepwise_need_first[["Added", "R²", "Adj. R²", "ΔR²", "p (ΔR²)"]].style.format(
    {"R²": "{:.3f}", "Adj. R²": "{:.3f}", "ΔR²": "{:+.3f}", "p (ΔR²)": "{:.3g}"})

## 3. All-subsets regression

All 63 possible combinations of the six variables are estimated. For each number of variables the best model (highest adjusted R²) is shown. Adjusted R² and AIC both penalise extra variables, so they show whether an extra variable is worth adding. The highest VIF in each model is shown as a check on multicollinearity (VIF > 5 is a warning sign).

In [ ]:
rows = []
for k in range(1, len(ORDER) + 1):
    for combo in itertools.combinations(ORDER, k):
        ols, params, pvalues = fit(combo)
        rows.append({"k": k,
                     "Variables": " + ".join(VARIABLES[v] for v in combo),
                     "Adj. R²": ols.rsquared_adj, "AIC": ols.aic,
                     "Max VIF": max_vif(combo),
                     "All significant": all(pvalues[v] < 0.05 for v in combo)})

subsets = pd.DataFrame(rows).sort_values("Adj. R²", ascending=False)
subsets.to_csv(PROCESSED + "all_subsets_results.csv", index=False)

best_per_k = subsets.loc[subsets.groupby("k")["Adj. R²"].idxmax()].set_index("k")
best_per_k.style.format({"Adj. R²": "{:.3f}", "AIC": "{:.1f}", "Max VIF": "{:.2f}"})

In [ ]:
# Top 10 of all 63 models
subsets.head(10).style.format({"Adj. R²": "{:.3f}", "AIC": "{:.1f}", "Max VIF": "{:.2f}"}).hide(axis="index")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(subsets["k"], subsets["Adj. R²"], color="#b0b0b0", s=20, label="All 63 models")
ax.plot(best_per_k.index, best_per_k["Adj. R²"], color="#1d639b", marker="o", label="Best model per size")
for k, row in best_per_k.iterrows():
    ax.annotate(f"{row['Adj. R²']:.3f}", (k, row["Adj. R²"]), textcoords="offset points",
                xytext=(0, 8), ha="center", fontsize=8)
ax.set_xlabel("Number of explanatory variables")
ax.set_ylabel("Adjusted R²")
ax.set_title(f"All-subsets regression of charging point density (N = {len(data)})")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIGURES + "figure_all_subsets.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Composite indices

Two pairs of variables are so strongly correlated that they largely measure the same thing. Instead of dropping one variable of each pair, the two are combined into one index: the average of their z-scores.

- **Urbanity (need)** = population density (log) + share of apartments
- **Progressive profile** = share of highly educated residents + left-wing vote share

In [ ]:
print("r (log density, apartments):", round(data["log_population_density"].corr(data["meergezinswoning_pct"]), 2))
print("r (education, left-wing vote share):", round(data["hoogopgeleid_pct"].corr(data["linkse_stemmen_pct"]), 2))

z["urbanity_index"] = z[["log_population_density", "meergezinswoning_pct"]].mean(axis=1)
z["progressive_index"] = z[["hoogopgeleid_pct", "linkse_stemmen_pct"]].mean(axis=1)
# Standardise the indices again so their β is comparable with the other variables
for col in ["urbanity_index", "progressive_index"]:
    z[col] = (z[col] - z[col].mean()) / z[col].std()

VARIABLES.update({"urbanity_index": "Urbanity index", "progressive_index": "Progressive index"})
COMPOSITE = ["med_gestandaardiseerd_inkomen_keur", "ouderen_65plus_pct", "urbanity_index", "progressive_index"]

ols, params, pvalues = fit(COMPOSITE)
X = sm.add_constant(z[COMPOSITE]).values
composite = pd.DataFrame({
    "β": params[COMPOSITE],
    "p-value": pvalues[COMPOSITE],
    "VIF": [variance_inflation_factor(X, i) for i in range(1, X.shape[1])],
}, index=COMPOSITE)
composite.index = [VARIABLES[v] for v in composite.index]
composite.loc["Adj. R²", "β"] = ols.rsquared_adj
composite.loc["N", "β"] = ols.nobs
composite.to_csv(PROCESSED + "composite_model_results.csv")
composite.round(3)

## 5. Summary

The tables and figures above are used in section 7 of the report:

- **Hierarchical regression (section 2):** which group adds the most explained variation, and which effects disappear when need is added.
- **Order check (section 2b):** how much income adds on top of need.
- **All-subsets (section 3):** how few variables are needed to reach the highest adjusted R².
- **Composite indices (section 4):** the result when correlated pairs are combined instead of dropped.

> **TODO:** write 3–5 sentences interpreting the results above, in the same style as notebook 7.